In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

Trevor Marting R implementation can be found at  https://github.com/trevormartin/papers/blob/master/community2vec/community2vecpaper.r

- 60.504 articles
- 8.172.305 comments
- 21 Topic 1
- 172 Topic 2
- 1.222 Topic 3 (+1 topic with some issue)
    - 46.194 unique authors on articles with Topic 3 tags
    - 1.084.012 author-topic relantionships
        - 182.943 author-topic presence after cutoff
    - 1.072 topic are retained (topics with no comment overlap with other topics are dropped)
    - 500 context topics with most unique users (MIGHT BE TOO LARGE)


# 1. Load 2018 data

In [2]:
base_path = Path("data")
articles = pd.read_csv(
    base_path / "articles_file_18.tsv",
    sep="\t",
    dtype={"Article_Id": "string", "Topic_2": "string"},
    keep_default_na=False,
)

comment_frames = [
    pd.read_csv(
        comment_path,
        sep="\t",
        dtype={"Article_Id": "string", "Author_Id": "string"},
        keep_default_na=False,
        usecols=["Article_Id", "Author_Id"],
    )
    for comment_path in sorted(base_path.glob("comments_file_*_18.tsv"))
]
comments = pd.concat(comment_frames, ignore_index=True)

print(f"Memory usage of articles: {articles.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")
print(f"Number of articles: {len(articles)}")
display(articles.head())
print(f"Memory usage of comments: {comments.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")
print(f"Number of comments: {len(comments)}")
display(comments.head())
print(f"Unique Topic_1 tags: {articles['Topic_1'].nunique()}")
print(f"Unique Topic_2 tags: {articles['Topic_2'].nunique()}")
print(f"Unique Topic_3 tags: {articles['Topic_3'].nunique()}")


Memory usage of articles: 19.92 MB
Number of articles: 60504


,Article_Id,Topic_1,Topic_2,Topic_3,Topic_Labels,Timestamp
0,2000067810628,wissenschaft,forschungsmagazin,,"wissenschaft,Wissenschaft;forschungsmagazin,Fo...",1519455600
1,2000067820241,lifestyle,essentrinken,,"lifestyle,Lifestyle;essentrinken,Essen & Trinken",1515319200
2,2000067822106,lifestyle,essentrinken,,"lifestyle,Lifestyle;essentrinken,Essen & Trinken",1515059963
3,2000067845125,lifestyle,essentrinken,,"lifestyle,Lifestyle;essentrinken,Essen & Trinken",1514815201
4,2000067865508,lifestyle,uhren,,"lifestyle,Lifestyle;uhren,Uhren & Schmuck",1515232801


Memory usage of comments: 982.30 MB
Number of comments: 8172305


,Author_Id,Article_Id
0,88df487c6a36449b,2000063073063
1,7f677dde8db7f4c6,2000071292618
2,06aaa183c2ba2a52,2000071292618
3,e5019747c7f5efbf,2000071292618
4,b5ca5330cfdba804,2000063073063


Unique Topic_1 tags: 21
Unique Topic_2 tags: 172
Unique Topic_3 tags: 1223


# 2. Topic-User number of comments count

In [3]:
# Merge comments with articles to get the Topic_3 for each comment
comments_with_topics = comments.merge(articles[["Article_Id", "Topic_3"]], on="Article_Id", how="inner")

# Filter out comments with empty Topic_3 or Author_Id
comments_with_topics = comments_with_topics[
    comments_with_topics["Topic_3"].str.strip().ne("")
    & comments_with_topics["Author_Id"].str.strip().ne("")
]

# Rows are editorial topics; columns are users; values are comment counts.
topic_user_comment_counts = (
    comments_with_topics.assign(
        Topic_3=comments_with_topics["Topic_3"].str.strip(),
        Author_Id=comments_with_topics["Author_Id"].str.strip(),
    )
    .pivot_table(
        index="Topic_3",
        columns="Author_Id",
        values="Article_Id",
        aggfunc="size",
        fill_value=0,
    )
    .reindex(
        index=sorted(articles.loc[articles["Topic_3"].str.strip().ne(""), "Topic_3"].str.strip().unique()),
        fill_value=0,
    )
    .astype("int64")
)
topic_user_comment_counts.index.name = "Topic_3"
topic_user_comment_counts.columns.name = "Author_Id"

print(f"Matrix shape: {topic_user_comment_counts.shape}")
assert topic_user_comment_counts.to_numpy().sum() == len(comments_with_topics)

display(topic_user_comment_counts.head())


Matrix shape: (1222, 46194)


Author_Id,0000493a226805ea,0000852c2877c8e1,000102bf623d747d,00014b561b98cb37,000311a836a81192,0004e7adbc87e004,000ab2f8e91a8e02,000ab63c9958302b,000c61e955dd0a0f,000e10f91d3acfe8,...,fff3740c550799fd,fff442827c55ac4e,fff768034a4659a6,fff876660201cc51,fff93b130c22d9ab,fffb285b5e98c6c1,fffb63952b448c97,ffff957e387a4545,ffffdc6e4c0d833f,ffffe1007f6fb4bf
Topic_3,,,,,,,,,,,,,,,,,,,,,
abfall,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
abnehmbuecher,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
absolventen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
abtreibung,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
adcannes,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 3. Binary Matrix (cutoff: comments > 5)

Cutoff is aribitrary for now

In [4]:
CUTOFF = 5
topic_user_comment_counts_count = (topic_user_comment_counts > 0).to_numpy().sum()
print(f"Number of non-zero entries in topic-user comment count matrix: {topic_user_comment_counts_count}")

topic_user_presence_sparse = csr_matrix( # used for for memory efficiency
    (topic_user_comment_counts > CUTOFF).to_numpy(),
    dtype="int64",
)
topic_user_presence = pd.DataFrame.sparse.from_spmatrix( # used to keep labels
    topic_user_presence_sparse,
    index=topic_user_comment_counts.index,
    columns=topic_user_comment_counts.columns,
).astype(pd.SparseDtype("int64", fill_value=0))

topic_user_presence_count = topic_user_presence_sparse.nnz
print(f"Number of non-zero entries in topic-user presence matrix: {topic_user_presence_count}")

topic_user_presence_row_counts = topic_user_presence_sparse.sum(axis=1).A1
topics_with_no_comments_above_cutoff = topic_user_presence.index[topic_user_presence_row_counts == 0].tolist()
if topics_with_no_comments_above_cutoff:
    print(f"Number of topics with no comments above cutoff ({CUTOFF}): {len(topics_with_no_comments_above_cutoff)}")
    print(f"Topics with no comments above cutoff: {topics_with_no_comments_above_cutoff}")

print(f"Matrix shape: {topic_user_presence.shape}")
display(topic_user_presence.head())

Number of non-zero entries in topic-user comment count matrix: 1084012
Number of non-zero entries in topic-user presence matrix: 182943
Number of topics with no comments above cutoff (5): 148
Topics with no comments above cutoff: ['adcannes', 'agendalesen', 'algerien', 'allerheiligen-', 'angola', 'anschlagberlin', 'ansichtssachen-filmstarts', 'arselectronica', 'belvedere', 'bergmayr', 'bregenzer-festspiele', 'burkinafaso', 'burnout', 'cca', 'charliehebdo', 'csrspezial', 'cumhuriyetprozess', 'digitalezukunft', 'druesenhormone', 'dtm', 'ecuador', 'effie', 'einkochen', 'elsalvador', 'erdbebenpano', 'erling', 'erneuerbare-energie', 'esslmuseum', 'etatmaus', 'familieanderswo', 'fedcup', 'filmfestival-crossingeurope', 'filmfestival-locarno', 'filmpreise', 'fortbildung', 'google-etat', 'guantanamo', 'haiti', 'hcb', 'honduras', 'immo-einmaleins', 'immofinanzierung', 'immostudenten', 'indienpakistan', 'innereorgane', 'itpatente', 'journalistenunterdruck', 'judo', 'jugend', 'kamerun', 'kampl', '

Author_Id,0000493a226805ea,0000852c2877c8e1,000102bf623d747d,00014b561b98cb37,000311a836a81192,0004e7adbc87e004,000ab2f8e91a8e02,000ab63c9958302b,000c61e955dd0a0f,000e10f91d3acfe8,...,fff3740c550799fd,fff442827c55ac4e,fff768034a4659a6,fff876660201cc51,fff93b130c22d9ab,fffb285b5e98c6c1,fffb63952b448c97,ffff957e387a4545,ffffdc6e4c0d833f,ffffe1007f6fb4bf
Topic_3,,,,,,,,,,,,,,,,,,,,,
abfall,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
abnehmbuecher,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
absolventen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
abtreibung,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
adcannes,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 4. Co-ocurrence Matrix

- Main diagonal is set to zero to eliminate self-pairs
- Topics with no overlap (number of comments above threshold) with any other topic are not used

In [5]:
# Sparse dot product of topic-user presence matrix with its transpose to get co-occurrence counts
topic_cooccurrence_sparse = (topic_user_presence_sparse @ topic_user_presence_sparse.T).tocsr()

# Remove self-pairs.
assert (topic_cooccurrence_sparse.diagonal() == topic_user_presence_sparse.sum(axis=1).A1).all()
topic_cooccurrence_sparse.setdiag(0)
topic_cooccurrence_sparse.eliminate_zeros()

# Retain only topics that have at least one co-occurrence with another topic
has_overlap = np.asarray(topic_cooccurrence_sparse.sum(axis=1)).ravel() > 0
retained_topics = topic_user_presence.index[has_overlap]
topic_cooccurrence_sparse = (topic_cooccurrence_sparse[has_overlap][:, has_overlap]).tocsr()
print(f"Number of total topics: {len(topic_user_presence.index)}")
print(f"Number of topics with at least one co-occurrence with another topic: {len(retained_topics)}")

# Recover index and column label
topic_cooccurrence_counts = pd.DataFrame.sparse.from_spmatrix(
    topic_cooccurrence_sparse,
    index=retained_topics,
    columns=retained_topics,
).astype(pd.SparseDtype("int64", fill_value=0))
topic_cooccurrence_counts.index.name = "Topic_3"
topic_cooccurrence_counts.columns.name = "Topic_3"

print(f"Co-occurrence matrix shape: {topic_cooccurrence_counts.shape}")
assert (topic_cooccurrence_sparse.diagonal() == 0).all()
assert topic_cooccurrence_counts.equals(topic_cooccurrence_counts.T)

display(topic_cooccurrence_counts.head())

Number of total topics: 1222
Number of topics with at least one co-occurrence with another topic: 1072
Co-occurrence matrix shape: (1072, 1072)


Topic_3,abfall,abnehmbuecher,absolventen,abtreibung,adipositas0,admira,aegypten,aerzte-in-not,aethiopien,afghanistan,...,zeitsprung,zentralafrika,zoidl,zoomwochenschau,zoos,zsolt,zweitedeutscheliga,zweiteliga,zyklus,zypern
Topic_3,,,,,,,,,,,,,,,,,,,,,
abfall,0,2,3,8,15,1,4,4,1,11,...,2,1,1,3,3,3,2,1,6,1
abnehmbuecher,2,0,1,2,3,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
absolventen,3,1,0,4,5,1,2,2,1,2,...,1,1,1,1,2,1,1,2,2,1
abtreibung,8,2,4,0,14,1,9,5,1,12,...,1,1,1,5,4,1,2,3,18,1
adipositas0,15,3,5,14,0,2,5,6,2,2,...,1,1,1,4,2,1,1,2,10,1


# 5. Choose context topics

In [6]:
# rank topics by the amount of unique users that commented on them (above the cutoff)
topic_user_presence_row_counts = topic_user_presence_sparse.sum(axis=1).A1
topic_ranks = pd.Series(topic_user_presence_row_counts, index=topic_user_presence.index).sort_values(ascending=False)
print("Topics ranked by the amount of unique users that commented on them (above the cutoff):")
print(topic_ranks.head())

# filter co-occurrence matrix to keep only the top 500 topics by unique user count as columns
top_n = 500
top_topics = topic_ranks.head(top_n).index
filtered_topic_cooccurrence_counts = topic_cooccurrence_counts.loc[:, top_topics]
print(f"Filtered co-occurrence matrix shape (top {top_n} topics): {filtered_topic_cooccurrence_counts.shape}")
display(filtered_topic_cooccurrence_counts.head())

Topics ranked by the amount of unique users that commented on them (above the cutoff):
Topic_3
fpoe                      3689
rauscher                  3359
kommentare-der-anderen    3320
spoe                      3235
koalition                 3189
dtype: int64
Filtered co-occurrence matrix shape (top 500 topics): (1072, 500)


Topic_3,fpoe,rauscher,kommentare-der-anderen,spoe,koalition,usa,sozialpolitik,gesundheitspolitik,deutschland,arbeitsmarkt,...,ub-architektur-und-urbanistik,finnland,hagen,libyen,inklusion,erstebankopen,salzburger-festspiele,stephen-hawking,pohl,norwegen
Topic_3,,,,,,,,,,,,,,,,,,,,,
abfall,79,73,81,64,80,66,68,74,66,71,...,6,4,3,4,4,1,3,3,3,4
abnehmbuecher,3,3,4,2,3,3,2,3,3,3,...,1,2,1,1,2,1,1,1,1,1
absolventen,8,6,10,8,9,4,8,8,6,10,...,2,2,2,1,3,1,2,1,2,1
abtreibung,113,96,126,90,96,108,106,94,106,101,...,5,5,3,1,6,1,3,2,4,2
adipositas0,71,69,78,58,63,59,67,70,54,69,...,8,3,2,1,5,1,2,1,3,2


# 6. PPMI Transform

The PMI matrix assigns positive values to topic pairs whose co-occurrence exceeds what would be expected under independence, given their marginal frequencies; negative values if topics co-ocurr less than expected; and zero if its exactly as often as expected. PPMI keeps only the positive values, so negative relationships are ignored. The `log` base changes only the scale, not which associations are positive.

$$
\operatorname{PPMI}(i,j)
=
\max\left(
\ln \frac{P(i,j)}{P(i)\,P(j)},
\;0
\right)
=
\ln\left(
\frac{n_{ij}N}
{n_i n_j}
\right).
$$

$$
P(i)=\frac{n_i}{N},
\qquad
P(j)=\frac{n_j}{N},
\qquad
P(i,j)=\frac{n_{ij}}{N}.
$$

These probabilities are defined over the co-occurrence counts in the matrix after selecting context columns, rather than over distinct users:

- $P(i,j)$ is the probability that a randomly selected co-occurrence instance belongs to row topic $i$ and context topic $j$.
- $P(i)$ is the probability that a randomly selected co-occurrence instance belongs to row topic $i$, regardless of its context topic.
- $P(j)$ is the probability that a randomly selected co-occurrence instance belongs to context topic $j$, regardless of its row topic.
- $n_{ij}$ is the matrix entry for topic pair $(i,j)$: the number of distinct users exceeding the comment cutoff in both topics.
- $n_i = \sum_{b \in \mathcal{J}} n_{ib}$ is the sum of row $i$ across all selected context topics. It counts co-occurrence instances, not distinct users.
- $n_j = \sum_{a \in \mathcal{I}} n_{aj}$ is the sum of column $j$ across all retained row topics. It also counts co-occurrence instances, not distinct users.
- $N = \sum_{a \in \mathcal{I}}\sum_{b \in \mathcal{J}} n_{ab}$ is the sum of every entry in the context-filtered co-occurrence matrix, not the total number of users.

Here, $\mathcal{I}$ is the set of retained row topics and $\mathcal{J}$ is the set of selected context topics. A user can contribute to multiple topic pairs and therefore be counted multiple times in the marginal totals and in $N$.

Before calculating probabilities, add-one smoothing is applied to every entry, including zeros.


In [7]:
counts = (filtered_topic_cooccurrence_counts.sparse.to_coo().toarray().astype(np.float64))
counts += 1.0 # add 1 to every cell for smooothing

total_count = counts.sum() # N
row_totals = counts.sum(axis=1, keepdims=True) # n_i marginal sums for rows
column_totals = counts.sum(axis=0, keepdims=True) # n_j marginal sums for columns (context topics)

# Calculate PPMI matrix
pmi = np.log((counts * total_count) / (row_totals * column_totals))
ppmi = np.maximum(pmi, 0.0)

# Store the unnormalized PPMI matrix with index and column labels
ppmi_sparse = csr_matrix(ppmi)
unnormalized_ppmi = pd.DataFrame.sparse.from_spmatrix(
    ppmi_sparse,
    index=filtered_topic_cooccurrence_counts.index,
    columns=filtered_topic_cooccurrence_counts.columns,
)

# Normalize each vector to unit lenght
row_norms = np.linalg.norm(ppmi, axis=1, keepdims=True)
normalized_ppmi = np.divide(ppmi, row_norms, out=np.zeros_like(ppmi), where=row_norms > 0,)
topic_embeddings = normalized_ppmi

print(f"PPMI matrix shape: {unnormalized_ppmi.shape}")
print(f"Positive PPMI entries: {ppmi_sparse.nnz}")
display(unnormalized_ppmi.head())

PPMI matrix shape: (1072, 500)
Positive PPMI entries: 306238


Topic_3,fpoe,rauscher,kommentare-der-anderen,spoe,koalition,usa,sozialpolitik,gesundheitspolitik,deutschland,arbeitsmarkt,...,ub-architektur-und-urbanistik,finnland,hagen,libyen,inklusion,erstebankopen,salzburger-festspiele,stephen-hawking,pohl,norwegen
Topic_3,,,,,,,,,,,,,,,,,,,,,
abfall,0,0,0,0,0,0,0,0,0,0,...,0.444103,0.235222,0,0.244602,0.323507,0,0.081671,0.131438,0,0.261953
abnehmbuecher,0,0,0,0,0,0,0,0,0,0,...,1.080766,1.613822,1.182777,1.217738,1.702108,1.42981,1.27795,1.327717,1.126377,1.235088
absolventen,0,0,0,0,0,0,0,0,0,0,...,0.998543,1.126133,1.100553,0.730049,1.502101,0.942121,1.195726,0.840028,1.044153,0.747399
abtreibung,0,0,0.098512,0,0,0.043616,0.026424,0,0.005391,0,...,0.12353,0.251121,0,0,0.493557,0,0,0,0,0
adipositas0,0,0,0,0,0,0,0,0,0,0,...,0.769573,0.086233,0,0,0.579984,0,0,0,0.004253,0


In [8]:
# Inspect the strongest and weakest topic pairs in the unnormalized PPMI matrix
ppmi_pairs = (unnormalized_ppmi.rename_axis(index="topic", columns="context_topic").stack().rename("ppmi").reset_index())
ppmi_pairs = ppmi_pairs[ # exlude self-pairs and zeros
    (ppmi_pairs["topic"] != ppmi_pairs["context_topic"])
    & (ppmi_pairs["ppmi"] > 0)
]

n_pairs = 20
highest_ppmi_pairs = ppmi_pairs.nlargest(n_pairs, "ppmi")
display(highest_ppmi_pairs)

,topic,context_topic,ppmi
520828,wimbledon,atpfinale,3.276234
394741,serbien,kosovo,3.264045
232815,kosovo,serbien,3.241142
23867,atpfinale,wimbledon,3.230296
221867,kitz,wimbledon,3.223858
158946,giro,radwm,3.155053
520732,wimbledon,australianopen,3.145601
23732,atpfinale,australianopen,3.101657
221995,kitz,erstebankopen,3.061033
27328,australianopen,atpfinale,3.042582


# 7. Exploration of embedding vectors

In [9]:
import plotly.express as px
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


topic_labels = unnormalized_ppmi.index.to_numpy()

tsne = TSNE(n_components=2, perplexity=30, init="pca", learning_rate="auto", random_state=42)
topic_embeddings_2d = tsne.fit_transform(topic_embeddings)

tsne_topics = pd.DataFrame(topic_embeddings_2d, index=topic_labels, columns=["tsne_1", "tsne_2"],)
tsne_topics = pd.DataFrame(topic_embeddings_2d,columns=["tsne_1", "tsne_2"])
tsne_topics["topic"] = topic_labels

fig = px.scatter(
    tsne_topics,
    x="tsne_1",
    y="tsne_2",
    hover_name="topic",
    hover_data={
        "topic": False,
        "tsne_1": ":.3f",
        "tsne_2": ":.3f",
    },
    title="t-SNE projection of topic embeddings",
    labels={
        "tsne_1": "t-SNE dimension 1",
        "tsne_2": "t-SNE dimension 2",
    },
)

fig.update_traces(marker={"size": 8, "opacity": 0.7})
fig.update_layout(width=1100, height=800, template="plotly_white")
fig.show()